<a href="https://colab.research.google.com/github/jppeirce/DSC210-Foundations-of-Data-Science/blob/main/Notes/07-exploratory_data_analysis/07-exploratory_data_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 7: Exploratory Data Analysis

**DSC 210 Foundations of Data Science**

References:
- [Hands-on Introduction to Data Science with Python](https://florian-huber.github.io/data_science_course/) (CC BY-NC-SA 4.0), especially the chapters on *Data Cleaning and Preprocessing* and *Statistical Measures and Distributions*
- [pandas documentation](https://pandas.pydata.org/docs/) and [seaborn documentation](https://seaborn.pydata.org/)
- John W. Tukey, *Exploratory Data Analysis* (1977), the book that gave this subject its name

```
ASK  ->  GET  ->  [ EXPLORE ]  ->  MODEL  ->  COMMUNICATE
```

In Module 6 we learned to **GET** data into a DataFrame and to select, filter, and sort it.
Today we put those tools to work on the stage they were built for: **EXPLORE**.

## Learning objectives

1. Explain the purpose of exploratory data analysis (EDA).
2. Distinguish between numerical, categorical, and datetime variables.
3. Use pandas to inspect the structure and contents of a dataset.
4. Compute and interpret descriptive statistics.
5. Create and interpret common visualizations.
6. Identify missing values, unusual observations, and potential data quality issues.
7. Begin asking meaningful questions about a dataset rather than immediately trying to model it.

Our running dataset for this module is a sample of **6,433 real taxi rides in New York City** from March 2019. One dataset, the whole module: by the end you will know these taxis uncomfortably well.

---
## 1. What is Exploratory Data Analysis?
---

# Spoiler Alert!

If you are going to work in the field of data science you might be surprised to find out that getting, understanding, and preparing the data you need for an analysis or the training of a model might cost you much more time, sweat and tears than the shiny parts that people will see in your final results.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/Notes/07-exploratory_data_analysis/fig_data_acquisition_problems.png?raw=true" width="800">



### 1.1 The temptation to skip ahead

Here is a situation you will be in many times in your career. Someone hands you a dataset and asks to build a model to answer a question.

For example, you are given a data set of of New York taxi rides and asked: *"Build a model that predicts fares."*

The temptation is to feed the data straight into a model. Resist it. At this moment you do not know:

- what a row in this dataset even represents,
- whether the columns mean what their names suggest,
- whether values are missing, duplicated, or impossible,
- whether the data can answer the question at all.

A model built on data you do not understand will happily produce numbers. They will just be the wrong numbers, delivered with great confidence.

Remember: <img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/Notes/07-exploratory_data_analysis/gigo.png?raw=true" width="400">

**Definition.** *Exploratory Data Analysis (EDA)* is the process of getting to know a dataset, through summaries, tables, and pictures, **before** drawing conclusions or building models.

The statistician John Tukey, who wrote the book on EDA in 1977, put the spirit of it this way: *"The greatest value of a picture is when it forces us to notice what we never expected to see."*

### 1.2 What EDA is, and what it is not

EDA **is**:

- asking simple questions and answering them with small computations,
- drawing quick, unpolished plots and actually looking at them,
- being professionally suspicious of every column,
- keeping a running list of findings and new questions.

EDA **is not**:

- proving a hypothesis (that is formal inference, later in the course),
- building a predictive model (that is the MODEL stage),
- making publication-quality graphics (that is COMMUNICATE).

The goal of EDA is *understanding*: a short list of what you learned, what worries you, and what you want to ask next.

Note: although EXPLORE has its own box in our pipeline, exploration actually happens at every stage. We explore when data arrives (do the types look right?), while cleaning (which values are suspicious?), while modeling (which points have outsized influence?), and while communicating (which picture tells the story?).

### 1.3 New York City taxi rides

For this module assume that you are a new data analyst at the NYC Taxi and Limousine Commission (TLC), the agency that regulates the city's taxis. The TLC publishes records of every taxi trip in the city; our dataset is a sample of **6,433 rides from March 2019** that ships with the seaborn library.

Two kinds of cabs appear in the data:
- **Yellow cabs** may pick up street hails anywhere in the city.
- **Green cabs** (the "boro taxis") were introduced to improve service outside the Manhattan core; they may only pick up street hails in upper Manhattan and the outer boroughs.

Each row is **one taxi ride**: when and where it started and ended, how far it went, what it cost, and how it was paid for.

### Activity 1

Your boss at the TLC asks: *"Are riders tipping less than they used to? Should we change fare policy?"*

Before we write a single line of code: with a partner, list **three things you would want to know about this dataset** before trusting any answer that comes out of it. Think about where the data came from, what exactly got recorded, and what could quietly go wrong.

Be ready to share one item with the class.

---
## 2. Loading and Inspecting Data
---

**Look before you leap:** The first fifteen minutes with any new dataset follows the same ritual.

In [ ]:
# RUN-TOGETHER
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')   # a clean default look for all plots

In [ ]:
# RUN-TOGETHER
# The dataset lives at a public URL, just like a file someone emailed you.
url = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/taxis.csv'
taxis = pd.read_csv(url)

# ALWAYS check that it loaded the way you expected.
taxis.head()

### 2.1 A guided tour of the columns

`head()` shows the first five rows. Lets investigate:

| Column | Meaning |
| --- | --- |
| `pickup`, `dropoff` | date and time the ride began and ended |
| `passengers` | number of riders in the cab |
| `distance` | trip length in miles |
| `fare` | metered fare, in dollars |
| `tip` | recorded tip, in dollars |
| `tolls` | bridge and tunnel tolls, in dollars |
| `total` | total amount charged, in dollars |
| `color` | `yellow` or `green` cab |
| `payment` | `credit card` or `cash` |
| `pickup_zone`, `dropoff_zone` | neighborhood names |
| `pickup_borough`, `dropoff_borough` | the NYC boroughs |

Even five rows generate observations. The `pickup` column *looks* like a date and time, but is it stored that way? Some tips are exactly 0. Hmmm...we should probably make a note of that.

In [ ]:
# RUN-TOGETHER
# head() shows the top of the file... which may not be typical of the file.
taxis.tail()

In [ ]:
# RUN-TOGETHER
# A RANDOM peek is more honest than the top or bottom.
# random_state pins the randomness so we all see the same five rows.
taxis.sample(5, random_state=210)

Why bother with `sample()`? Files often arrive sorted by date, by ID, or by something invisible to you. The first rows may all be from one day or one company. Five random rows is small, but may give a more accurate portrait.

**Quick look:** in your sample of five, is anything surprising? Any zeros, blanks, or odd combinations?

### Activity 2 (predict, then run)

Before running the next cell, write down your guesses:

a. How many **rows** does `taxis` have? (You have seen the number already today.)

b. How many **columns**?

Then fill in the blanks and check yourself. Recall from Module 6 that `shape` is an *attribute*, not a method: no parentheses.

In [ ]:
# Activity 2
print('shape (rows, cols):', taxis._shape___)
print()
print('columns:', list(taxis.____))

In [ ]:
# RUN-TOGETHER
taxis.info()

`info()` is dense, so read it top to bottom, three questions at a time:

1. **How much data?** 6,433 rows, 14 columns.
2. **Is anything missing?** Compare each Non-Null Count to 6,433. Look at `payment`: 6,389 non-null. That means **44 rides have no recorded payment type**. Same story for the zone and borough columns. We did not go hunting for missing data; `info()` volunteered it.
3. **How is each column stored?** The Dtype column says `pickup` and `dropoff` are stored as `str`. But they are clearly dates and times. A column can *look* like a date and be stored as text, and text cannot be subtracted, sorted by clock, or grouped by hour. We will fix this later.

In [ ]:
# RUN-TOGETHER
# Descriptive statistics for every NUMERIC column.
taxis.describe()

`describe()`: For each numeric column we get the count of non-missing values, the mean, the standard deviation, and the five-number summary (min, 25%, 50%, 75%, max). Note:

- **`fare`:** the mean is about 13.09 dollars but the median (the 50% row) is 9.50. When the mean sits well above the median, a long right tail of expensive rides is pulling it up. Remember from the elementary statistical...the mean chases extreme values, the median does not.
- **`fare` again:** the maximum is 150 dollars, more than fifteen times the median. Real, or an error? We do not know yet. Make a note of it.
- **`passengers`:** the minimum is **0**. A taxi ride with nobody in it? Write that down too.

Each oddity goes into our notebook as a *question*. Also notice what `describe()` did **not** show: the text columns. It summarizes numeric columns by default.

In [ ]:
# RUN-TOGETHER
# The same idea for the NON-numeric columns.
taxis.describe(exclude='number')

For text columns we get: `count` (non-missing), `unique` (number of distinct values), `top` (most common value), and `freq` (how often the top value appears). Manhattan is the top pickup borough with 5,268 of 6,407 recorded pickups; this sample is very Manhattan-heavy, which matters when we generalize.

Here is a small puzzle hiding in plain sight: `pickup_borough` has **4** unique values but `dropoff_borough` has **5**. Some borough receives rides but never produces them, at least in this sample.

### Activity 3

Using only commands from this section plus `value_counts()` from Module 6, answer:

a. How many rides are missing a `dropoff_zone`?

b. What was the largest `total` charged?

c. What is the **median** trip `distance`?

d. Which borough appears in `dropoff_borough` but never in `pickup_borough`?

One new tool for part (a): `taxis['col'].isna()` returns True/False for every entry, and chaining `.sum()` counts the Trues. (We come back to this.)

In [ ]:
# Activity 3
# a. missing dropoff_zone: total rows minus non-null count, or use isna
print('missing dropoff_zone:', taxis['dropoff_zone'].____().sum())

# b. largest total fares
print('largest total:', taxis['total'].____())

# c. median distance
print('median distance:', taxis['distance'].____())

# d. compare the categories of the two borough columns
print(taxis['pickup_borough'].value_counts())
print(taxis[____].value_counts())

---
## 3. Understanding Variables
---

`info()` told us how each column is *stored*. But to choose the right tool for a column, we need to know what kind of thing it *is*. Taking the average of `fare` makes sense. Taking the average of `pickup_borough` is nonsense. Counting the distinct values of `payment` is useful; counting the distinct values of `fare` is not. The type of a variable determines which questions are even well-posed.

### 3.1 Three families of variables

**Numerical (quantitative)** variables are measured amounts. Arithmetic on them is meaningful.

- *Continuous:* can take any value in a range: `distance`, `fare`, `tip`.
- *Discrete:* whole-number counts: `passengers`.

**Categorical (qualitative)** variables place each observation into a group.

- *Nominal:* groups with no natural order: `color`, `payment`, `pickup_borough`.
- *Ordinal:* groups with a natural order (small < medium < large). Our taxi data has no ordinal column. But it customers rated their ride: not satisfactory/satisfactory/great, then their ratings would be an ordinal categorical variable.

**Datetime** variables are points in time: `pickup`, `dropoff`. They deserve their own family because time has rich structure: order, durations, hours of the day, days of the week, seasons.

A rough matching of families to tools, which is really the table of contents for the rest of this module:

| Family | Typical summaries | Typical pictures |
| --- | --- | --- |
| Numerical | mean, median, spread, quantiles | histogram, boxplot |
| Categorical | counts, proportions | bar chart |
| Datetime | ranges, durations | counts over time |

### 3.2 The dtype is not destiny

The pandas dtype tells you how a column is *stored*, not what it *means*, and the two can disagree:

- `pickup` is stored as text but means a datetime.
- A ZIP code column would be stored as an integer but means a category (averaging ZIP codes is nonsense).
- `passengers` is stored as an integer and could reasonably be treated as a count *or* as a grouping category, depending on the question.

Part of EDA is noticing these mismatches and correcting the ones that block your analysis.

In [ ]:
# RUN-TOGETHER
taxis.dtypes

### Activity 4 (Think-Pair-Share)

With a partner, classify **every** column of `taxis` as numerical, categorical, or datetime, and flag any column that could defensibly go two ways.

Then take a side and defend it: is `passengers` numerical or categorical?

### 3.3 Repairing the datetime columns

Right now `taxis['dropoff'] - taxis['pickup']`, which *should* give the ride duration, would fail: you cannot subtract one string from another. The fix is `pd.to_datetime()`, which parses text into true datetime values.

In [ ]:
# RUN-TOGETHER
taxis['pickup'] = pd.to_datetime(taxis['pickup'])
taxis['dropoff'] = pd.to_datetime(taxis['dropoff'])

taxis.dtypes.head(3)   # confirm the repair took

Now the payoff. With genuine datetimes we can *derive new columns* (the Module 6 move) that unlock whole families of questions: How long did rides take? What time of day do people ride? Which day of the week is busiest?

The `.dt` accessor reaches into a datetime column and pulls out parts: `.dt.hour`, `.dt.day_name()`, and more.

In [ ]:
# RUN-TOGETHER
# Duration in minutes: subtract, then convert the result to seconds, then minutes.
taxis['duration_min'] = (taxis['dropoff'] - taxis['pickup']).dt.total_seconds() / 60

# Extract the hour (0-23) and the weekday name of each pickup.
taxis['pickup_hour'] = taxis['pickup'].dt.hour
taxis['pickup_day'] = taxis['pickup'].dt.day_name()

taxis[['pickup', 'duration_min', 'pickup_hour', 'pickup_day']].head()

Three new columns, three new dimensions of the dataset, and none of them existed in the file.

**A large part of EDA is manufacturing the variable you actually want from the ones you were given.**

### Activity 5

Before you run anything: guess the median ride duration in minutes, and guess the busiest day of the week. Then check:

a. Compute the **median** of `duration_min`.

b. What **proportion** of rides carry 2 or more passengers? Use the trick that the mean of a True/False column is the proportion of Trues (True counts as 1, False as 0).

c. Which day of the week has the most rides?

In [ ]:
# Activity 5
# a. median duration
print('median duration (min):', taxis['duration_min'].____())

# b. proportion of shared rides: mean of a boolean column
print('share with 2+ passengers:', (taxis['passengers'] ____ 2).mean().round(3))

# c. rides per weekday
print(taxis[____].value_counts())

---
## 4. Looking at One Variable at a Time
---

We now know what the columns are. The next step is to understand each interesting variable **on its own** before we study how variables relate. Statisticians call this *univariate* analysis, and doing it first is a discipline: if you do not know what `fare` looks like alone, you cannot recognize when something strange happens to it in a scatterplot.

The plan follows our variable families: categorical first (frequency tables and bar charts), then numerical (summary statistics, histograms, boxplots), then our derived time variables.

### 4.1 Categorical variables: frequency tables

For a categorical column the fundamental question is: **how often does each category occur?** `value_counts()` from Module 6 answers it, and `normalize=True` converts counts to proportions.

In [ ]:
# RUN-TOGETHER
print(taxis['color'].value_counts())
print()
print(taxis['payment'].value_counts(normalize=True).round(3))

Yellow cabs dominate the sample (5,451 of 6,433, about 85%), and roughly 72% of rides with a recorded payment were paid by credit card.

One caution: by default `value_counts()` silently **drops missing values**. The payment proportions above are proportions *of the 6,389 recorded payments*, not of all 6,433 rides. Passing `dropna=False` would show the missing group as its own row.

A frequency table becomes a **bar chart** when you want your audience to compare sizes at a glance.

In [ ]:
# RUN-TOGETHER
sns.countplot(data=taxis, x='pickup_borough',
              order=taxis['pickup_borough'].value_counts().index)
plt.title('Rides by pickup borough')
plt.show()

The bar chart is the frequency table drawn as a picture, with the bars sorted so the eye can rank them instantly. The story: this dataset overwhelmingly describes **Manhattan** pickups. Whatever we conclude about "NYC taxis" today is mostly a conclusion about Manhattan taxis, with a side of Queens. Keep that in the findings list.

### 4.2 Numerical variables: histograms

For a numerical column, a frequency table of exact values is useless (almost every `fare` is different). Instead we ask about the **distribution**: which ranges of values are common, which are rare, and what shape the whole thing makes.

The **histogram** answers this by chopping the number line into bins and counting how many observations land in each.

**Predict first:** sketch, in the air or on paper, the shape you expect for `distance`. Symmetric bell? Flat? Lopsided?

In [ ]:
# RUN-TOGETHER
sns.histplot(data=taxis, x='distance')
plt.title('Distribution of trip distance')
plt.show()

Not a bell. The mass piles up below about 3 miles and a long tail stretches to the right, out past 30. This shape is called **right-skewed** (the tail points right).

Skew explains a mystery from Section 2: mean distance is 3.02 miles but the median is 1.64. The handful of very long trips drag the mean upward while the median stands firm. For skewed variables, *median and quartiles are usually the more honest summary*.

In [ ]:
# RUN-TOGETHER
sns.histplot(data=taxis, x='fare')
plt.title('Distribution of fare')
plt.show()

`fare` tells the same right-skewed story: mean 13.09 above median 9.50, long expensive tail. But look closely around **50 dollars**: a little bump interrupts the smooth decay. Fares of about 52 dollars are more common than fares of 40 or 60. A smooth process should not do that.

Write it down as a question: *why would many unrelated rides cost almost exactly the same, fairly large amount?*

#### The bins are a choice

A histogram depends on how many bins you cut the axis into, and the default is generally nobody's friend. Watch the same variable at two extremes.

In [ ]:
# RUN-TOGETHER
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=taxis, x='fare', bins=10, ax=axes[0])
axes[0].set_title('fare, 10 bins')
sns.histplot(data=taxis, x='fare', bins=100, ax=axes[1])
axes[1].set_title('fare, 100 bins')
plt.show()

Ten bins melt the 52-dollar bump away entirely; one hundred bins make it unmistakable but shred the rest into noise. Neither is "the truth". The practical habit: **always try two or three bin counts** before trusting what a histogram tells you.

### 4.3 Boxplots: the five-number summary as a picture

The histogram shows the whole shape; the **boxplot** compresses a distribution into its landmarks

- the line inside the box is the **median**,
- the box spans the **first to third quartile** (the middle 50% of the data, whose width is the interquartile range, IQR),
- the whiskers extend to the most extreme points within **1.5 × IQR** of the box,
- anything beyond the whiskers is drawn as an individual **flagged point**.

In [ ]:
# RUN-TOGETHER
sns.boxplot(data=taxis, x='fare')
plt.title('Boxplot of fare')
plt.show()

Same variable, same story, new lens: the box is squashed against the left (half of all fares live between 6.50 and 15 dollars) while a parade of flagged points marches off to the right.

Are those flagged points errors? **Not necessarily.**

Note: the histogram shows *shape*, the boxplot shows *landmarks and flags*. We will usually want both.

### 4.4 The time variables

Our derived columns are variables too, and they answer a question every city planner cares about: **when do people take taxis?**

In [ ]:
# RUN-TOGETHER
sns.countplot(data=taxis, x='pickup_hour')
plt.title('Rides by hour of day')
plt.show()

Read it like a day in the life of the city: the quietest hours are 3 to 5 in the morning (about 50 to 70 rides each), demand climbs through the morning, and the peak is the evening, 6 to 8 pm, at roughly 400 rides per hour.

And here is EDA doing its job, because a good plot generates the *next* question: the 4 and 5 am rides are few, but are they **different in kind**? Who takes a taxi at 4:30 am?

### Activity 6

Your turn to run the full univariate playbook on a fresh variable: **`tip`**.

a. Compute its descriptive statistics with `describe()`.

b. Draw a histogram.

c. Draw a boxplot.

d. In two sentences, in your own words: what shape is this distribution, what is a typical tip, and is there anything odd?

There *is* something odd. When you find it, write it in your notes exactly as a question. We will solve it in Section 5.

In [ ]:
# Activity 6
# a. summary statistics
print(taxis['tip'].____())

# b. histogram
sns.histplot(data=taxis, x=____)
plt.show()

# c. boxplot
sns.____(data=taxis, x='tip')
plt.show()

# d. Your two sentences, as a comment:
#

---
## 5. Looking at Relationships
---

Every variable has now had its solo. But the questions people actually pay data scientists to answer live **between** variables: *What drives the price of a ride? Do the boroughs differ? Who tips?*

Let us take the fare question, as a working example. Candidate explanations worth testing:

1. Longer trips cost more (`distance` vs `fare`).
2. More passengers cost more (`passengers` vs `fare`).
3. Prices differ by borough (`pickup_borough` vs `fare`).

Each candidate pairs two variables, and the *types* of the pair determine the tool.
- Numerical with numerical: scatterplot.
- Categorical with numerical: group summaries.
- Categorical with categorical: crosstab.

### 5.1 Two numerical variables: the scatterplot

A **scatterplot** puts one variable on each axis and one dot per observation. It is the single most information-dense plot in this course: shape, strength, direction, and exceptions, all in one picture.

In [ ]:
# RUN-TOGETHER
sns.scatterplot(data=taxis, x='distance', y='fare', s=10, alpha=0.4)
plt.title('Fare vs distance')
plt.show()

What can we say? Fares climb with distance in a tight, roughly linear band. The meter charges by distance (plus time), and the data shows it.

But the exceptions are where EDA requires us to focus. Three features to point at:

- A **horizontal stripe near 52 dollars**, running across many distances. The same fare for wildly different trip lengths: this is our histogram bump from Section 4, now with more evidence. It looks like a *flat rate*. (It is: in 2019 trips between Manhattan and JFK airport charged a flat 52 dollars. Our dataset contains 131 rides at exactly that fare, almost all touching an airport.)
- **Dots pinned to distance 0** with fares well above zero. Zero-mile rides that cost money. Hmm...that is interesting. We should investigate further.
- A few extreme points in the far upper right: 30-plus-mile, 150-dollar rides.

One plot, three discoveries, and we did not fit a single model.

### Activity 7 (predict, then run)

Hypothesis 2: fuller cabs cost more. It sounds plausible; more people, more service.

**Predict with your neighbor first:** what will the scatterplot of `passengers` vs `fare` look like if the hypothesis is true? What if it is false? Then fill in the blanks and run.

In [ ]:
# Activity 7
sns.scatterplot(data=taxis, x=____, y=____, s=10, alpha=0.4)
plt.title('Fare vs passengers')
plt.show()

Note: *a plausible story died on contact with data, in ten seconds, for free.* This is why we explore before we model.

### 5.2 Correlation: one number for a relationship

Our eyes say distance-and-fare is "tight" and passengers-and-fare is "nothing". The **correlation coefficient** r turns that judgment into a number between -1 and 1:

- r near **+1**: as one variable rises, the other reliably rises (tight upward line).
- r near **-1**: as one rises, the other reliably falls.
- r near **0**: no *linear* relationship.

We will treat r intuitively for now (the formula can wait). Three warnings before we compute a single one, because correlation is the most misused number in data science:

1. r sees only **straight-line** patterns. A perfect U-shape can score r = 0.
2. r is **not causation**. Ice cream sales correlate with drownings; summer causes both.
3. r can be dragged around by a few extreme points, and we know our data has some.

In [ ]:
# RUN-TOGETHER
print('distance vs fare:      ', taxis['distance'].corr(taxis['fare']).round(2))
print('passengers vs fare:    ', taxis['passengers'].corr(taxis['fare']).round(2))
print('distance vs tip:       ', taxis['distance'].corr(taxis['tip']).round(2))

The numbers agree with our eyes: 0.92 for distance and fare, 0.01 for passengers and fare, and a middling 0.45 for distance and tip.

To survey all the numeric pairs at once, `.corr()` on a set of columns produces a **correlation matrix**, and a heatmap makes it readable.

In [ ]:
# RUN-TOGETHER
numeric_cols = ['passengers', 'distance', 'fare', 'tip', 'tolls', 'total']
corr_matrix = taxis[numeric_cols].corr().round(2)

sns.heatmap(corr_matrix, annot=True, cmap='vlag', vmin=-1, vmax=1)
plt.title('Correlations among numeric variables')
plt.show()

Reading a correlation matrix is its own small skill:

- The diagonal is all 1s (everything correlates perfectly with itself); ignore it.
- `fare` and `total` correlate at 0.97. Impressive? No: **`total` is computed from `fare`** (fare plus tip, tolls, and surcharges). Correlations that are baked in by arithmetic tell you about bookkeeping, not about the world. Spotting them is part of understanding your columns.
- `passengers` correlates with nothing. An entire row of noise.
- The interesting cluster is distance, fare, and tip, all moderately-to-strongly linked.

### Activity 8 (Think-Pair-Share)

Two quick arguments to evaluate with a partner:

a. A classmate says: "tolls and distance correlate at 0.64, so paying tolls causes trips to be longer." What is wrong with this sentence, and what is the more sensible story?

b. Earlier we wondered whether 4 and 5 am rides are *different in kind*. Propose a group summary (in words, not code) that would check whether early-morning rides are unusually long. Then, if time allows, compute the mean `distance` for each `pickup_hour` and find the peak.

In [ ]:
# Activity 8b (optional code)
taxis.groupby('pickup_hour')['distance'].mean().round(2)

### 5.3 A categorical and a numerical variable: group summaries

Hypothesis 3 asked whether boroughs differ. The tool is `groupby()` from Module 6: split the rides by borough, then summarize a numeric column within each group.

In [ ]:
# RUN-TOGETHER
taxis.groupby('pickup_borough')[['distance', 'fare', 'duration_min']].mean().round(2)

Now the borough differences have numbers: Queens pickups average 7.5 miles and about 25 dollars, triple Manhattan's 2.3 miles and 11 dollars. Before reaching for exotic explanations, look at a map: **both major airports sit in Queens**. Airport runs are long, and they dominate that borough's average.

One caution that applies to every group summary: check the group *sizes*. The Bronx mean rests on only 99 rides; Manhattan's on 5,268. Small groups make jumpy averages.

#### The tipping mystery, solved

Time to cash in the oddity from Activity 6: 36% of rides record a tip of exactly zero. Are New Yorkers really that cold?

Here is a hypothesis a student raises every year: maybe it depends on **how people pay**. That is a categorical-numerical question, and we now own the tool for it.

In [ ]:
# RUN-TOGETHER
print(taxis.groupby('payment')['tip'].mean().round(2))
print()
print('zero-tip rides by payment type:')
print(taxis[taxis['tip'] == 0]['payment'].value_counts())

Look at that top line: the mean tip on cash rides is not merely small. It is **0.00. Every single one of the 1,812 cash rides records a tip of exactly zero.**

Real populations do not behave like that; 1,812 people do not unanimously tip nothing. When a value is *impossibly* consistent, be do not blame the people. Instead we investigate the measurement.
>The actual explanation: the taxi meter only records tips paid by card.

Cash tips happen, in the back seat, off the books; the dataset simply never sees them. (The TLC documents this limitation of its trip records.)

Pause on what just happened, because it is the most important moment of the module:

- The value 0 is not "missing". It is not flagged. It passes every format check. The data is, in a bookkeeping sense, perfectly clean, **and it still lies**.
- A model trained naively on this data would learn that cash customers never tip, and someone might make a policy decision downstream of that artifact.
- No algorithm catches this. It was caught by a person who looked at a suspicious spike, asked why, split the data by a category, and knew a little about how taxi meters work.

That is exploratory data analysis.

### Activity 9

The average *recorded* tip is 2.19 dollars in yellow cabs but only 0.80 in green cabs. A colleague drafts the headline: **"Green cab riders tip far less."**

Given what you just learned, and one more fact (41% of green-cab rides are paid in cash versus 26% of yellow-cab rides), write two sentences to your colleague. Is the headline justified?

### 5.4 Two categorical variables: crosstabs

That activity leaned on a claim: green cabs carry more cash rides. Where did that number come from? From the tool for pairs of categorical variables, the **cross-tabulation**: a grid of counts for every combination of two categories.

In [ ]:
# RUN-TOGETHER
print(pd.crosstab(taxis['color'], taxis['payment']))
print()
# normalize='index' turns each ROW into proportions that sum to 1
print(pd.crosstab(taxis['color'], taxis['payment'], normalize='index').round(2))

The raw counts are hard to compare because there are five times as many yellow rides, which is exactly why the normalized version exists: within green cabs, 41% of rides are cash; within yellow cabs, 26%. A crosstab is just `value_counts()` grown up to two dimensions, and `normalize` chooses which direction we compare in.

### 5.5 Comparing distributions across groups

Group *means* compress each group to one number. Sometimes you want to compare whole distributions, and this is where the boxplot becomes a team sport: one box per group, sharing an axis.

In [ ]:
# RUN-TOGETHER
sns.boxplot(data=taxis, x='pickup_borough', y='fare')
plt.title('Fare by pickup borough')
plt.show()

The Queens box floats higher and stretches wider, telling us its fares are not just bigger on average but more variable (a mix of local trips and airport runs). Manhattan's box is low and tight with a long fringe of flagged points. One picture, four distributions, directly comparable.

### Activity 10

We close this section with a famous warning label. Below we load 13 different datasets, stacked in one file: each has an `x` column, a `y` column, and a `dataset` label (A through M).

Here is the twist, and it is not a trick question: **all 13 datasets have essentially identical summary statistics.** Same means, same standard deviations, same correlation, to two decimal places. Your job is to find out whether they are, therefore, the same data.

In [ ]:
# RUN-TOGETHER
# Load the file and relabel the datasets A, B, C, ...
ds_url = ('https://raw.githubusercontent.com/florian-huber/data_science_course/'
          'a595bd6b19565cedff0c3917012be6e05223f7fb/datasets/datasaurus.csv')
saurus = pd.read_csv(ds_url)

new_names = 'ABCDEFGHIJKLM'
convert = {old: new_names[i] for i, old in enumerate(saurus['dataset'].unique())}
saurus['dataset'] = saurus['dataset'].map(convert)

print(saurus['dataset'].unique())
saurus.head(3)

In [ ]:
# Activity 10a
# Group by dataset and compare count, mean, and std of x and y across the 13 groups.
summary = saurus.____('dataset')[['x', 'y']].agg(['count', 'mean', 'std']).round(2)
summary

Confirm it with your own eyes: 13 rows, and the summary columns are all but identical. If `describe()`-style numbers were the whole story, these datasets would be interchangeable.

**Predict before running the next cell:** will the scatterplots look the same too?

In [ ]:
# Activity 10b
# One scatterplot per dataset, in a grid.
sns.relplot(data=saurus, x='x', y='y', col='dataset', col_wrap=4, height=2)
plt.show()

A dinosaur. A star. Concentric circles. Parallel stripes. Thirteen wildly different pictures wearing identical statistical uniforms.

The moral, and it deserves to be written on the wall of every data science lab: **summary statistics compress, and compression destroys information. Always, always plot your data.** (This collection is the "Datasaurus Dozen" of Matejka and Fitzmaurice, 2017, a modern descendant of Anscombe's famous quartet from 1973.)

Connect it back to our taxis: which of today's discoveries would summary statistics alone have missed? The 52-dollar stripe. The zero-distance rides. Arguably the tip spike. The plots did the finding; the statistics confirmed the details.

---
## 6. Detecting Data Quality Problems
---

All module long, oddities have been piling up in our notes:
- 44 missing payments,
- rides with zero passengers,
- zero-mile rides that cost 120 dollars.

A working EDA quality sweep has four checks: **missing values, duplicates, impossible values, and outliers.** Be begin by talking about ways to *detect and document*. Deciding what to do about each problem (delete? fill in? keep?) is the subject of the next module on data cleaning, and good decisions there depend entirely on the careful detective work here.

### 6.1 Missing values

`info()` hinted at missing data back in Section 2. The direct tool is `isna()`, which returns True/False for every cell, and column sums turn those into counts.

In [ ]:
# RUN-TOGETHER
print(taxis.isna().sum())
print()
print('rows with at least one missing value:', taxis.isna().any(axis=1).sum())

Missingness is confined to `payment` (44) and the four location columns (26 to 45 each), touching 92 rows in all, about 1.4% of the data. Nothing is missing in the money or distance columns.

Counting is not understanding, though. Before deciding anything, **look at the actual afflicted rows**: do they seem otherwise normal, or is missingness a symptom of a deeper problem?

In [ ]:
# RUN-TOGETHER
taxis[taxis['payment'].isna()].head()

These rows look like ordinary rides that simply lack a payment entry. That is mildly reassuring, but the question that actually matters is: **is the missingness random, or systematic?** If the 44 missing payments were all from one cab company or one neighborhood, dropping them would quietly bias every conclusion downstream. With 44 rows we cannot fully settle it today, but noticing that the question exists is the point.

In the next module on cleaning we will discuss deletion strategies and imputation, with their trade-offs.

### Activity 11 (Think-Pair-Share)

Which is the more damaging hole in this dataset: the **44 missing `payment`** values, or the **45 missing `dropoff_zone`** values?

Argue it both ways before you settle. (Hint: damaging *for which analysis?*)

### 6.2 Duplicates

Large data pipelines (especially accross multiple sources or collaborators) can create duplicate rows.  A file gets submitted twice, a merge goes sideways, a sensor hiccups.

In [ ]:
# RUN-TOGETHER
print('duplicated rows in taxis:', taxis.duplicated().sum())

# A tiny demonstration of what duplicated() catches:
demo = pd.DataFrame({'ride': [1, 2, 2, 3], 'fare': [7.0, 9.5, 9.5, 12.0]})
print()
print(demo)
print()
print(demo.duplicated())          # True marks the SECOND appearance
print()
print(demo.drop_duplicates())     # one line to remove them

Our taxi sample comes back clean: zero exact duplicates. Two footnotes worth saying aloud:

- `duplicated()` flags rows where **every** column matches. Two different rides that coincidentally share a fare are not duplicates; two rows agreeing on all 14 fields, including pickup time to the second, almost certainly are.
- A clean result is still a result. "I checked for duplicates and found none" belongs in your findings.

### 6.3 Impossible and suspicious values

Missing data announces itself. The more dangerous problems are values that are *present, well-formatted, and impossible*. No function called `find_impossible_values()` exists, because impossibility depends on what the data means. What we have instead are **boolean masks** (Module 6) plus common sense about taxis.

In our previous EDA we saw a few potential "ref flags:"
- rides with no passengers,
- rides with no distance but a fare, and
- rides that end before they begin.

Let's count these up.

In [ ]:
# RUN-TOGETHER
print('rides with 0 passengers:            ', (taxis['passengers'] == 0).sum())
print('0-distance rides that cost money:   ',
      ((taxis['distance'] == 0) & (taxis['fare'] > 0)).sum())
print('rides with non-positive duration:   ', (taxis['duration_min'] <= 0).sum())

In [ ]:
# RUN-TOGETHER
# Look at the worst offenders: zero-distance rides, most expensive first.
zero_dist = taxis[taxis['distance'] == 0]
zero_dist.sort_values('fare', ascending=False)[
    ['pickup', 'distance', 'fare', 'total', 'pickup_zone', 'dropoff_zone']
].head()

The tally: 96 phantom-passenger rides, 51 zero-mile rides with fares reaching 120 dollars, 6 rides with non-positive duration.

What *are* these? We do not know, and it is important to be comfortable saying so. Plausible stories include a driver forgetting to enter the passenger count, a meter started and cancelled but still charged, a GPS failure recording no distance, or clock errors. The data cannot tell us which.

What do we do about these rows of data? Not a fix but a *documented inventory*: which rows, how many, and the plausible explanations. Whether to drop, repair, or keep them is a cleaning decision, and the right choice will depend on our analysis. For a fare model, a 120-dollar zero-mile ride may be toxic to the model and analysis.

### Activity 12

Investigate the luxury end: rides with a **fare above 100 dollars**.

a. How many are there?

b. Display their `distance`, `fare`, `pickup_zone`, and `dropoff_zone`.

c. In a comment: which of these look like real (if pricey) rides, and which look like data problems? What tips you off?

In [ ]:
print(expensive.total)

In [ ]:
# Activity 12
expensive = taxis[taxis['fare'] > 100]

# a. how many?
print('rides over $100:', expensive.____[0])

# b. inspect the relevant columns
expensive[[____, ____, 'pickup_zone', 'dropoff_zone']]

# c. your verdict as a comment:
#

### 6.4 Outliers: an introduction

The zero-distance rides were *impossible*. But what about that 36.7-mile, 150-dollar ride? Nothing about it is impossible; it is just far from everything else. Values like this are called **outliers**: observations distant from the bulk of the data.

Flagged outlier points come in at least three kinds, with three different fates:

1. **Errors** (the 120-dollar zero-mile ride): fix or remove, during cleaning.
2. **True but rare values** (a real ride to the far edge of Queens): keep; deleting them fabricates a tidier world than the one you live in.
3. **A hidden subpopulation** (the 52-dollar flat-rate stripe): indicates and interesting finding that should be looked at carefully.

A common flagging heuristic is the rule from Section 4. Compute the quartiles and the interquartile range, then flag anything outside the fences:

- lower fence = Q1 - 1.5 × IQR
- upper fence = Q3 + 1.5 × IQR

In [ ]:
# RUN-TOGETHER
q1 = taxis['fare'].quantile(0.25)
q3 = taxis['fare'].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
print('fences:', round(lower, 2), 'to', round(upper, 2))

flagged = taxis[(taxis['fare'] < lower) | (taxis['fare'] > upper)]
print('flagged rides:', flagged.shape[0],
      '=', round(100 * flagged.shape[0] / taxis.shape[0], 1), '% of the data')

The rule flags 592 rides, nine percent of the dataset, as "outliers", including every airport trip in the sample. (The lower fence is negative, so nothing is flagged on the cheap side; on right-skewed data this is common.)

Nine percent of a dataset is not a fringe; it is a constituency. If we mechanically deleted the flagged rows, we would erase airport travel from our picture of NYC taxis and then wonder why our fare model fails every traveler with a suitcase. So the workflow is: the rule *flags*, a human *investigates*, and any removal happens later, deliberately, with reasons written down.

### Activity 13

Run the same IQR analysis on **`distance`**:

a. Compute the fences and the number of flagged rides.

b. Display the 5 longest rides with their zones.

c. In a comment: are the longest rides errors or real? What in the other columns supports your call?

In [ ]:
# Activity 13
q1 = taxis['distance'].quantile(____)
q3 = taxis['distance'].quantile(____)
iqr = q3 - q1
upper = q3 + 1.5 * iqr
print('upper fence:', round(upper, 2), 'miles')
print('flagged:', (taxis['distance'] > ____).sum())

# b. the 5 longest rides
taxis.sort_values(____, ascending=____)[
    ['distance', 'fare', 'pickup_zone', 'dropoff_zone']
].head()

# c. your verdict:
#

---
## 7. EDA as an Iterative Process
---

In this module had a few unsepected reults. We found these results by followed a **loop**:

```
    ask a question
        |
        v
  compute or plot something
        |
        v
  notice something unexpected   ->   (write it down!)
        |
        v
  refine the question, ask a new one
        |
        v
       repeat
```

Watch the loop run through what we actually did:

- *What do fares look like?* -> histogram -> **a bump at 52 dollars** -> *why?* -> scatterplot -> a flat-rate stripe -> the JFK flat fare.
- *What do tips look like?* -> histogram -> **a spike at zero** -> *who tips zero?* -> group by payment -> **cash tips are never recorded** -> a measurement artifact that changes how we read every tip number in the dataset.
- *When do people ride?* -> counts by hour -> **a 4-to-5 am trickle** -> *are those rides different?* -> mean distance by hour -> pre-dawn rides are twice as long -> airport runs.

Nobody could have written those questions down in advance, because each one was *created by the previous answer*. That is why EDA cannot be fully automated, and why it rewards curiosity more than memorized commands.

### 7.1 What we now know: a findings memo

Every EDA session should end with written findings, because understanding that stays in your head can evaporate in a week. Ours, in brief:

1. The dataset covers 6,433 NYC taxi rides from March 2019; one row per ride; heavily weighted toward Manhattan pickups (82%) and yellow cabs (85%).
2. Fares, distances, tips, and durations are all right-skewed; medians (9.50 dollars, 1.6 miles, 11 minutes) describe a typical ride far better than means.
3. Fare rises almost linearly with distance (r = 0.92); the number of passengers is irrelevant to price (r = 0.01).
4. A cluster of 131 rides at exactly 52 dollars reflects the Manhattan-JFK flat fare, not an anomaly.
5. Cash tips are structurally recorded as zero, so every tip statistic describes card-paying riders only, and cross-group tip comparisons (like yellow versus green) are contaminated by payment mix.
6. Queens pickups are longest and priciest (airports); pre-dawn rides are disproportionately long for the same reason.
7. Quality inventory: 92 rows (1.4%) have missing payment or location fields; 0 duplicate rows; 96 zero-passenger rides; 51 zero-distance rides with positive fares; 6 non-positive durations.
8. The 1.5 × IQR rule flags about 9 to 11% of rides on fare and distance, nearly all of them legitimate long trips; deletion by rule would erase airport travel.

Notice how many of them would change how we would build a fare model.

### 7.2 The hand-off: from exploring to cleaning and modeling

EDA does not end an analysis; it *sets the agenda* for the next stages.

**For the cleaning module (next),** our findings convert directly into a to-do list of decisions: what to do with 44 missing payments (drop? impute? keep as a category?), whether zero-distance and zero-passenger rides are salvageable or must go, whether to recompute or discard the 6 broken durations, and whether flat-rate JFK rides should be treated as their own category. Every one of those is a judgment call.

**For modeling (later),** EDA has already scouted the terrain:
- distance will carry a fare model almost by itself;
- hour and borough may add additional information; passengers adds nothing; and
- any model of tips must first reckon with "cash tips are structurally recorded as zero" or it will not be a good model.

```
ASK  ->  GET  ->  EXPLORE  ->  MODEL  ->  COMMUNICATE
 ^                                            |
 |________ findings raise new questions ______|
```

The pipeline diagram runs left to right, but the arrows secretly point both ways. You will come back and explore again after cleaning, and again after the first bad model. Every return trip is faster, because now you know the data.

### 7.3 A first-hour checklist for any new dataset

The EDA loop cannot be scripted, but the opening moves can.

1. Load the data and look at `head()`, `tail()`, and a random `sample()`.
2. Check `shape`: how much data do you actually have?
3. Read `info()`: missing values? wrong dtypes?
4. Read `describe()` for numeric columns and `describe(exclude='number')` for the rest; circle anything odd.
5. Classify each variable: numerical, categorical, datetime. Repair mis-typed columns and derive the ones you need.
6. For each key variable alone: `value_counts()` and bar charts (categorical); histogram, boxplot, and summary statistics (numerical). Try more than one bin width.
7. For key pairs: scatterplots, correlations, group summaries, crosstabs.
8. Hunt for trouble: missing values, duplicates, impossible values, extreme values.
9. Write down findings **and** open questions as you go, not at the end.
10. Only then start deciding what to clean and what to model.

### Activity 14

Your first real assignment from the TLC: *"We want to know whether green cabs serve different kinds of trips than yellow cabs."*

That is a real, open-ended question, exactly as vague as they come on the job. Work through the parts below with a partner; the blanks get sparser as you go, until the last part is all yours.

In [ ]:
# Activity 14, Part A: isolate and size up the green rides
green = taxis[taxis['color'] == ____]

print('green rides:', green.shape[0], 'of', taxis.shape[0])
print('as a share:', round(green.shape[0] / taxis.shape[0], 3))

In [ ]:
# Activity 14, Part B: where do green cabs pick up, compared to yellow?
# (Section 4 tools: value_counts with normalize, one per color)
print('GREEN pickups by borough:')
print(green['pickup_borough'].value_counts(normalize=True).round(2))
print()
print('YELLOW pickups by borough:')
yellow = taxis[taxis['color'] == ____]
print(yellow[____].value_counts(normalize=True).round(2))

In [ ]:
# Activity 14, Part C: do trip characteristics differ?
# Compare distance, fare, and duration across the two colors in ONE table.
taxis.groupby(____)[['distance', 'fare', 'duration_min']].____().round(2)

In [ ]:
# Activity 14, Part D: compare the DISTRIBUTIONS of distance, not just the means.
# One boxplot per color, sharing an axis (Section 5.5).
sns.boxplot(data=taxis, x=____, y=____)
plt.title('Trip distance by cab color')
plt.show()

In [ ]:
# Activity 14, Part E: quality check within the green subset, YOUR way.
# At minimum: missing values per column, and how many zero-distance rides.
# (No blanks here. You have all the tools.)

**Part F.** In a new markdown cell, write your memo to the TLC:

- **three findings**, each one sentence, each backed by a number or plot from Parts A through E;
- **one caution** about a comparison this data cannot fairly make (Section 5 gave you a big one);
- **one next question** you would investigate with more time or more data.

This structure (findings, cautions, next questions) is the report format of working data scientists. Sentence quality counts; you are writing for a boss, not a grader.

---
## References and further reading

- Tukey, J. W. (1977). *Exploratory Data Analysis*. Addison-Wesley. (The origin of the subject; skim a few pages just to see hand-drawn stem-and-leaf plots.)
- Huber, F. *Hands-on Introduction to Data Science with Python*: the chapters on [data cleaning and preprocessing](https://florian-huber.github.io/data_science_course/) and statistical measures and distributions extend Sections 4 and 6 of this module.
- Matejka, J., and Fitzmaurice, G. (2017). *Same Stats, Different Graphs: Generating Datasets with Varied Appearance and Identical Statistics through Simulated Annealing*. CHI 2017. (The Datasaurus Dozen paper.)
- Anscombe, F. J. (1973). Graphs in statistical analysis. *The American Statistician*, 27(1), 17-21.
- The [pandas user guide](https://pandas.pydata.org/docs/user_guide/index.html) and the [seaborn tutorial](https://seaborn.pydata.org/tutorial.html).
- NYC Taxi and Limousine Commission, [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page): the source of our sample, including data dictionaries that document (among other things) the cash-tip limitation.

*Next module: data cleaning and preprocessing, where our to-do list from Section 7.2 gets worked through, one decision at a time.*